In [1]:
from neo4j import GraphDatabase
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.generation.prompts import ERExtractionTemplate
from dotenv import load_dotenv
import os, time, asyncio, glob, csv

In [2]:
load_dotenv()

True

In [3]:
username = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")
URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
database = os.getenv("NEO4J_DATABASE", "neo4j")
AUTH = (username, password)

In [4]:
driver = GraphDatabase.driver(URI, auth=AUTH)

In [5]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [7]:
llm = OpenAILLM(model_name="gpt-4o-mini", api_key=OPENAI_API_KEY)
dimensions = 1536
embedder = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

In [8]:
entities = [
    {"label": "Person", "properties": [{"name": "name", "type": "STRING"}]},
    {"label": "House", "properties": [{"name": "name", "type": "STRING"}]},
    {"label": "Planet", "properties": [{"name": "name", "type": "STRING"}]}
]

relations = [
    {"label": "PARENT_OF", "source": "Person", "target": "Person"},
    {"label": "HEIR_OF", "source": "Person", "target": "House"},
    {"label": "RULES", "source": "House", "target": "Planet"},
]

In [19]:
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.experimental.components.pdf_loader import PdfLoader

# Create a PDF loader
pdf_loader = PdfLoader()

pipeline = SimpleKGPipeline(
    driver=driver,
    llm=llm,
    embedder=embedder,
    entities=entities,
    relations=relations,
    from_pdf=True,  # This is already default, but being explicit
    pdf_loader=pdf_loader,
)

In [20]:
async def run_pipeline_on_file(file_path, pipeline):
    await pipeline.run_async(file_path=file_path)

In [21]:
pdf_files = ["/Users/romul/Docs/ufs/pesquisa_graphrag/data/Dune (novel) - Wikipedia.pdf"]

for pdf_file in pdf_files:
    await run_pipeline_on_file(pdf_file, pipeline)

/Users/romul/Docs/ufs/pesquisa_graphrag/.venv/lib/python3.11/site-packages/pypdf/generic/_base.py:892: RuntimeWarning: coroutine 'run_pipeline_on_file' was never awaited
  return NameObject(ret)


In [22]:
from neo4j_graphrag.indexes import create_vector_index

create_vector_index(
    driver, 
    name="chunkEmbeddings", 
    label="Chunk",
    embedding_property="embedding",
    dimensions=1536,
    similarity_fn="cosine"
)

In [23]:
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever, Text2CypherRetriever
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.schema import get_schema

In [25]:
vector_retriever = VectorRetriever(
    driver,
    index_name="chunkEmbeddings",
    embedder=embedder,
    return_properties=["text"]
)

query = "Who is Paul Atreides?"
result = vector_retriever.search(query_text=query, top_k=10)

In [27]:
import pandas as pd

result_table = pd.DataFrame([(
    item.metadata['score'], 
    item.content[10:80],
    item.metadata['id']
    ) for item in result.items],
    columns=['Score', 'Content', 'ID']
)

In [28]:
result_table

,Score,Content,ID
0,0.928485,"nd), has instructed Paul in the waysof politic...",4:7f6208ea-e3c5-480a-949e-3cc18e50b00f:77
1,0.927582,their jihad could consume the entire universe....,4:7f6208ea-e3c5-480a-949e-3cc18e50b00f:78
2,0.923611,"life cycle, and his imagination was stimulated...",4:7f6208ea-e3c5-480a-949e-3cc18e50b00f:76
3,0.922230,"y party, armed like theothers. After becoming ...",4:7f6208ea-e3c5-480a-949e-3cc18e50b00f:83
4,0.922042,Gesserit wife of Count Fenring\nFremen\nThe Fr...,4:7f6208ea-e3c5-480a-949e-3cc18e50b00f:79
5,0.913980,the downfall of a galactic empire to Edward Gi...,4:7f6208ea-e3c5-480a-949e-3cc18e50b00f:82
6,0.906516,sert-sea; similarly all life on Earth is belie...,4:7f6208ea-e3c5-480a-949e-3cc18e50b00f:80
7,0.904606,"e in March 2011.[127]\nIn November 2016, Legen...",4:7f6208ea-e3c5-480a-949e-3cc18e50b00f:89
8,0.902526,"unfair to Herbert, anotherworking author, if h...",4:7f6208ea-e3c5-480a-949e-3cc18e50b00f:86
9,0.902193,"result of ""painful and slow personal progress....",4:7f6208ea-e3c5-480a-949e-3cc18e50b00f:84


In [29]:
rag = GraphRAG(llm=llm, retriever=vector_retriever)

In [30]:
print(rag.search(query).answer)

Paul Atreides is the main character of the novel "Dune" by Frank Herbert. He is the son of Duke Leto Atreides and Lady Jessica, and he is trained in warfare, political intrigue, and Bene Gesserit disciplines. Throughout the story, Paul discovers his prophetic powers and becomes a leader among the Fremen, the native inhabitants of the desert planet Arrakis. He is prophesied to be the "Lisan al-Gaib" or messiah, leading the Fremen in a holy war and ultimately taking control of the Imperium. Paul adopts the Fremen name Muad'Dib and navigates complex political dynamics involving his family's enemies, including House Harkonnen and the Emperor.


In [ ]:
# company_risk_list_query = """
# WITH node
# MATCH (node)-[:FROM_DOCUMENT]-(d:Document)-[:FIELD]-(c:Company)-:[FACES_RISK]-(rf:RiskFactor)
# RETURN c.name AS company, node.text AS context,
# collect(DISTINCT r.name) AS risks
# """

In [31]:
dune_universe_query = """
MATCH (p:Person)-[:HEIR_OF]->(h:House)-[:RULES]->(planet:Planet)
OPTIONAL MATCH (p)-[:PARENT_OF]->(child:Person)
RETURN p.name AS person, 
       h.name AS house, 
       planet.name AS planet,
       collect(DISTINCT child.name) AS children
ORDER BY p.name
"""

In [32]:
vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    retrieval_query=dune_universe_query
)

In [33]:
result = vector_cypher_retriever.search(
    query_text=query,
    top_k=10
)

for item in result.items:
    print(item.content[:50])

In [35]:
result = GraphRAG(llm=llm, retriever=vector_cypher_retriever)

print(rag.search(query_text=query).answer)

Paul Atreides is the son of Duke Leto Atreides and Lady Jessica, a key character in Frank Herbert's Dune series. He undergoes extensive training in combat and political intrigue from a young age and possesses prophetic abilities that are further enhanced by exposure to the spice on the desert planet Arrakis. As the story progresses, Paul becomes a revered leader among the Fremen, known as Muad'Dib, and is seen as a messianic figure destined to lead them in a holy war across the universe.


In [38]:
schema = get_schema(driver)
schema

'Node properties:\nDocument {embedding: LIST, id: STRING, source: STRING, text: STRING, path: STRING, createdAt: STRING}\nChunk {embedding: LIST, index: INTEGER, text: STRING}\nPerson {name: STRING}\nPlanet {name: STRING}\nHouse {name: STRING}\nRelationship properties:\n\nThe relationships:\n(:Chunk)-[:FROM_DOCUMENT]->(:Document)\n(:Chunk)-[:NEXT_CHUNK]->(:Chunk)\n(:Person)-[:HEIR_OF]->(:House)\n(:Person)-[:HEIR_OF]->(:Person)\n(:Person)-[:HEIR_OF]->(:Planet)\n(:Person)-[:PARENT_OF]->(:Person)\n(:Person)-[:PARENT_OF]->(:Planet)\n(:Person)-[:PARENT_OF]->(:House)\n(:Person)-[:FROM_CHUNK]->(:Chunk)\n(:Person)-[:RULES]->(:Person)\n(:Person)-[:RULES]->(:Planet)\n(:Person)-[:RULES]->(:House)\n(:Planet)-[:FROM_CHUNK]->(:Chunk)\n(:Planet)-[:PARENT_OF]->(:Person)\n(:Planet)-[:PARENT_OF]->(:Planet)\n(:Planet)-[:RULES]->(:Planet)\n(:House)-[:FROM_CHUNK]->(:Chunk)\n(:House)-[:RULES]->(:Planet)\n(:House)-[:RULES]->(:Person)\n(:House)-[:PARENT_OF]->(:House)\n(:House)-[:HEIR_OF]->(:House)'

In [39]:
query = "Who are the Fremen?"

text2cypher_retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=schema
)

cypher_query = text2cypher_retriever.get_search_results(query)
cypher_query.metadata["cypher"]

"cypher\nMATCH (person:Person)-[:RULES]->(planet:Planet {name: 'Fremen'})\nRETURN person.name\n"

In [44]:
import re

# Extract the Cypher query from the metadata
cypher_string = cypher_query.metadata["cypher"]

# Use regex to extract content after "cypher\n" or just clean the string
cypher_pattern = r"^cypher\n(.*)$"
match = re.match(cypher_pattern, cypher_string, re.DOTALL)

if match:
    extracted_cypher = match.group(1).strip()
else:
    extracted_cypher = cypher_string.strip()

print("Extracted Cypher Query:")
print(extracted_cypher)

Extracted Cypher Query:
MATCH (person:Person)-[:RULES]->(planet:Planet {name: 'Fremen'})
RETURN person.name


In [46]:
result = driver.execute_query(extracted_cypher)

In [47]:
for record in result.records:
    print(record)

In [49]:
from yfiles_jupyter_graphs import GraphWidget

def show_graph():
    driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"])
    )
    session = driver.session()
    widget = GraphWidget(graph=session.run("MATCH (s)-[r: !MENTIONS]->(t) RETURN s,r,t").graph())
    widget.node_label_mapping = "id"

    return widget

In [50]:
show_graph()

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: MENTIONS)} {position: line: 1, column: 16, offset: 15} for query: 'MATCH (s)-[r: !MENTIONS]->(t) RETURN s,r,t'


GraphWidget(layout=Layout(height='800px', width='100%'))